<a href="https://colab.research.google.com/github/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 1 — *Hello RL World*: Procesos de Decisión de Markov en código

**Curso:** Aprendizaje por Refuerzo: Fundamentos y Aplicaciones
**Institución:** Universidad Austral — Facultad de Ingeniería, Posgrados
**Docente:** Dr. Darío Ezequiel Díaz
**Fecha:** 5 de mayo de 2026

---

### Propósito de este cuaderno

Este primer cuaderno cumple dos funciones complementarias. Por un lado, **materializa en código** la formalización matemática del Proceso de Decisión de Markov $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ presentada en la sesión sincrónica. Por el otro, **introduce la interfaz Gymnasium**, marco computacional estándar sobre el cual edificaremos los algoritmos de las clases sucesivas.

El recorrido procede del particular al general: comenzaremos implementando un GridWorld desde cero —donde cada componente del MDP queda explícitamente expuesto— para luego acceder a un entorno canónico de Gymnasium (CartPole-v1) y observar cómo la abstracción algorítmica oculta esos mismos componentes detrás de una API uniforme.

### Hoja de ruta

| Sección | Contenido |
|:-:|---|
| 0 | Configuración del entorno computacional |
| 1 | El MDP en código: GridWorld desde cero |
| 2 | Gymnasium: el ecosistema estándar de entornos RL |
| 3 | Primera política: el agente aleatorio |
| 4 | Análisis estadístico de retornos |
| 5 | Anticipo: hacia el control óptimo (Clase 2) |
| — | Ejercicios propuestos |

> **Nota sobre la audiencia.** Este cuaderno asume familiaridad con probabilidad básica, álgebra lineal y programación en Python. La presentación es deliberadamente formal, con notación matemática rigurosa, conforme al perfil de los asistentes.

## 0. Configuración del entorno computacional

Comenzamos por garantizar que el entorno disponga de las dependencias necesarias. Las celdas siguientes funcionan tanto en una instalación local como en Google Colab; en este último caso, las bibliotecas se instalan en la sesión activa.

In [ ]:
# --- Detección de Google Colab e instalación de dependencias ---
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab ya trae numpy y matplotlib; instalamos sólo gymnasium
    !pip install -q gymnasium==1.3.0
    print("Entorno Google Colab detectado. Dependencias instaladas.")
else:
    print("Entorno local detectado. Asegúrese de tener instalado:")
    print("  pip install gymnasium==1.3.0 numpy matplotlib")

In [ ]:
# --- Importaciones y verificación de versiones ---
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from typing import Callable, List, Dict

# Reproducibilidad: fijamos una semilla global para los experimentos del cuaderno
SEMILLA = 42
rng_global = np.random.default_rng(SEMILLA)

print(f"NumPy      : {np.__version__}")
print(f"Matplotlib : {plt.matplotlib.__version__}")
print(f"Gymnasium  : {gym.__version__}")
print(f"Semilla fijada en: {SEMILLA}")

## 1. El MDP en código: GridWorld desde cero

### 1.1. Recordatorio formal

Recuérdese que un Proceso de Decisión de Markov es una quíntupla
$$
\mathcal{M} = (\mathcal{S},\, \mathcal{A},\, P,\, R,\, \gamma)
$$
donde $\mathcal{S}$ es el espacio de estados, $\mathcal{A}$ el espacio de acciones, $P(s' \mid s, a)$ la dinámica de transición, $R(s, a)$ la función de recompensa y $\gamma \in [0,1]$ el factor de descuento.

Implementaremos un **GridWorld** $4 \times 4$ con la siguiente especificación, idéntica a la presentada en la sesión sincrónica:

- $\mathcal{S} = \{(i, j) : i, j \in \{0, 1, 2, 3\}\} \setminus \text{muros}$, con muros en $(1, 1)$ y $(2, 1)$.
- Estado inicial: $s_0 = (0, 0)$. Estado terminal (meta): $s_g = (3, 3)$.
- $\mathcal{A} = \{\uparrow, \downarrow, \leftarrow, \rightarrow\}$ codificadas como $\{0, 1, 2, 3\}$.
- $P$ es **determinista** en esta versión inicial: la acción se ejecuta con probabilidad uno, salvo que conduzca fuera de la grilla o contra un muro, en cuyo caso el agente permanece en su sitio.
- $R(s, a) = -1$ por cada paso, $+10$ al alcanzar la meta.
- $\gamma = 0.95$.

La opción de comenzar con un entorno determinista responde a una decisión pedagógica: facilita la verificación manual de las trayectorias y aísla el formalismo del MDP de la complejidad estocástica, que incorporaremos en clases ulteriores.

In [ ]:
class GridWorld:
    '''
    Implementación pedagógica de un MDP de tipo GridWorld 4x4.

    La clase expone explícitamente los cinco componentes del MDP
    (S, A, P, R, gamma) y respeta una API minimalista inspirada en Gymnasium:
    `reset()` para reiniciar y `step(a)` para avanzar un paso.
    '''

    # --- Constantes de codificación de acciones ---
    ARRIBA, ABAJO, IZQUIERDA, DERECHA = 0, 1, 2, 3
    NOMBRE_ACCION = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def __init__(self,
                 dim: int = 4,
                 muros: set = None,
                 inicio: tuple = (0, 0),
                 meta: tuple = (3, 3),
                 recompensa_paso: float = -1.0,
                 recompensa_meta: float = 10.0,
                 gamma: float = 0.95):

        self.dim = dim
        self.muros = set(muros) if muros is not None else {(1, 1), (2, 1)}
        self.inicio = inicio
        self.meta = meta
        self.recompensa_paso = recompensa_paso
        self.recompensa_meta = recompensa_meta
        self.gamma = gamma

        # Espacio de estados: todas las casillas excepto los muros
        self.S = [(i, j) for i in range(dim) for j in range(dim)
                  if (i, j) not in self.muros]
        self.A = [self.ARRIBA, self.ABAJO, self.IZQUIERDA, self.DERECHA]

        # Estado actual (se establece en reset)
        self.s_actual = None

    # ------------------------------------------------------------------
    # Componentes explícitos del MDP
    # ------------------------------------------------------------------
    def transicion(self, s: tuple, a: int) -> tuple:
        '''Función de transición determinista: P(s_prima | s, a) = 1 para s_prima = T(s, a).'''
        i, j = s
        if a == self.ARRIBA:
            candidato = (i, j + 1)
        elif a == self.ABAJO:
            candidato = (i, j - 1)
        elif a == self.IZQUIERDA:
            candidato = (i - 1, j)
        elif a == self.DERECHA:
            candidato = (i + 1, j)
        else:
            raise ValueError(f"Acción inválida: {a}")

        # Si el candidato está fuera de la grilla o es muro, el agente no se mueve
        if (candidato in self.muros or
            not (0 <= candidato[0] < self.dim) or
            not (0 <= candidato[1] < self.dim)):
            return s
        return candidato

    def recompensa(self, s: tuple, a: int, s_prima: tuple) -> float:
        '''Función de recompensa R(s, a, s_prima).'''
        if s_prima == self.meta:
            return self.recompensa_meta
        return self.recompensa_paso

    def es_terminal(self, s: tuple) -> bool:
        return s == self.meta

    # ------------------------------------------------------------------
    # API tipo Gymnasium
    # ------------------------------------------------------------------
    def reset(self) -> tuple:
        self.s_actual = self.inicio
        return self.s_actual

    def step(self, a: int) -> tuple:
        '''Devuelve (s_prima, r, terminado).'''
        s = self.s_actual
        s_prima = self.transicion(s, a)
        r = self.recompensa(s, a, s_prima)
        self.s_actual = s_prima
        return s_prima, r, self.es_terminal(s_prima)

    # ------------------------------------------------------------------
    # Utilidad para visualización
    # ------------------------------------------------------------------
    def renderizar(self, ax=None, trayectoria: list = None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(4.5, 4.5))

        # Grilla de fondo
        for i in range(self.dim):
            for j in range(self.dim):
                color = 'white'
                if (i, j) in self.muros:
                    color = '#4A4A4A'
                elif (i, j) == self.inicio:
                    color = '#A8DADC'
                elif (i, j) == self.meta:
                    color = '#F2A06B'
                ax.add_patch(plt.Rectangle((i, j), 1, 1,
                                            facecolor=color, edgecolor='#1E3A5F', lw=1.5))

        # Trayectoria si se provee
        if trayectoria is not None and len(trayectoria) > 1:
            xs = [s[0] + 0.5 for s in trayectoria]
            ys = [s[1] + 0.5 for s in trayectoria]
            ax.plot(xs, ys, '-', color='#D86A2C', lw=2, alpha=0.8)
            ax.plot(xs, ys, 'o', color='#D86A2C', markersize=6)

        # Etiquetas
        ax.text(self.inicio[0] + 0.5, self.inicio[1] + 0.5, 'S',
                ha='center', va='center', fontsize=14, fontweight='bold')
        ax.text(self.meta[0] + 0.5, self.meta[1] + 0.5, 'G',
                ha='center', va='center', fontsize=14, fontweight='bold')

        ax.set_xlim(0, self.dim)
        ax.set_ylim(0, self.dim)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
        return ax

# Instanciamos el entorno
gw = GridWorld()
print(f"|S| = {len(gw.S)} estados (excluyendo muros)")
print(f"|A| = {len(gw.A)} acciones")
print(f"gamma = {gw.gamma}")
print()
print("Espacio de estados:")
print(np.array(gw.S))

In [ ]:
# Visualización del entorno
fig, ax = plt.subplots(figsize=(5, 5))
gw.renderizar(ax)
ax.set_title('GridWorld 4x4 — entorno didáctico', fontsize=12, color='#1E3A5F')
plt.tight_layout()
plt.show()

### 1.2. Verificación de la propiedad de Markov

Recuérdese que la propiedad de Markov establece
$$
\mathbb{P}(S_{t+1} \mid S_0, A_0, \ldots, S_t, A_t) \;=\; \mathbb{P}(S_{t+1} \mid S_t, A_t).
$$

En nuestro GridWorld, la transición $T(s, a)$ depende **exclusivamente** del par $(s, a)$ actual; el método `transicion` no recibe ni consulta el historial. Esto garantiza por construcción la propiedad de Markov. Verifiquémoslo numéricamente: si llamamos a `transicion(s, a)` con dos historiales distintos pero idéntico estado-acción presente, debemos obtener idéntico resultado.

In [ ]:
# Demostración: la transición es función del par (s, a) y nada más
s_actual = (1, 2)
a_actual = gw.DERECHA

# Llamamos múltiples veces; el resultado debe ser idéntico
resultados = [gw.transicion(s_actual, a_actual) for _ in range(5)]
assert len(set(resultados)) == 1, "¡La propiedad de Markov falla!"
print(f"T({s_actual}, {gw.NOMBRE_ACCION[a_actual]}) = {resultados[0]}")
print(f"Resultado constante en {len(resultados)} llamadas.")
print("Propiedad de Markov verificada por construcción.")

## 2. Gymnasium: el ecosistema estándar de entornos RL

### 2.1. Filosofía de la API

[Gymnasium](https://gymnasium.farama.org/) es la evolución mantenida del histórico OpenAI Gym. Provee una **interfaz uniforme** para entornos heterogéneos: desde mundos discretos como FrozenLake hasta simulaciones físicas como MuJoCo. Esta uniformidad es deliberada: permite escribir algoritmos agnósticos al entorno y, recíprocamente, comparar entornos bajo un mismo agente.

La API se reduce conceptualmente a tres llamadas:

| Llamada | Propósito |
|---|---|
| `env.reset()` | Inicia un nuevo episodio. Devuelve el estado inicial $s_0$. |
| `env.step(a)` | Aplica la acción $a$. Devuelve $(s_{t+1}, r_{t+1}, \text{terminated}, \text{truncated}, \text{info})$. |
| `env.close()` | Libera recursos del simulador. |

> **Sutileza versión 1.x.** En las versiones recientes de Gymnasium, `step` distingue entre `terminated` (el episodio finalizó por dinámica del entorno) y `truncated` (se alcanzó un límite externo de pasos). Esta separación, ausente en Gym clásico, es crucial al estimar valores con bootstrapping.

### 2.2. CartPole-v1 como MDP

Trabajaremos con [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), problema canónico de control. Físicamente, consiste en un péndulo invertido montado sobre un carro que se desplaza a lo largo de un riel. El agente debe mantener el péndulo en posición vertical aplicando fuerzas laterales al carro.

Su formalización como MDP:

- $\mathcal{S} \subset \mathbb{R}^4$, donde $s = (x, \dot{x}, \theta, \dot{\theta})$ son posición y velocidad del carro, ángulo y velocidad angular del péndulo.
- $\mathcal{A} = \{0, 1\}$: empujar a la izquierda o a la derecha.
- $P$ es la dinámica determinista del sistema mecánico (con condición inicial aleatoria).
- $R(s, a) = +1$ por cada paso en que el péndulo se mantenga en pie.
- Episodio termina si $|\theta| > 12°$ o $|x| > 2.4$ (caída) o se cumplen 500 pasos (truncamiento).

In [ ]:
# Instanciación del entorno
env = gym.make('CartPole-v1')

print("=== Inspección del entorno CartPole-v1 ===")
print()
print(f"Espacio de observación  : {env.observation_space}")
print(f"  Forma                  : {env.observation_space.shape}")
print(f"  Cota inferior          : {env.observation_space.low}")
print(f"  Cota superior          : {env.observation_space.high}")
print()
print(f"Espacio de acciones      : {env.action_space}")
print(f"  Cardinalidad           : {env.action_space.n}")
print()
print(f"Límite de pasos por episodio: {env.spec.max_episode_steps}")

### 2.3. Mapeo formal MDP $\leftrightarrow$ API Gymnasium

Conviene fijar la correspondencia entre los símbolos matemáticos y los nombres del código, pues será invocada implícitamente en todos los algoritmos del curso:

| Notación matemática | Nombre en código |
|:-:|:-:|
| $s_0 \sim \mu_0$ | `obs, info = env.reset()` |
| $s_{t+1} \sim P(\cdot \mid s_t, a_t)$ | primer retorno de `env.step(a)` |
| $r_{t+1} = R(s_t, a_t, s_{t+1})$ | segundo retorno de `env.step(a)` |
| $\mathbb{1}\{s \text{ es terminal}\}$ | `terminated` |
| $a_t \in \mathcal{A}$ | argumento entero de `env.step` |

Examinemos un solo paso para fijar la sintaxis:

In [ ]:
obs, info = env.reset(seed=SEMILLA)
print(f"Estado inicial s_0 = {np.round(obs, 4)}")
print(f"  Componentes : (x, x_punto, theta, theta_punto)")
print(f"  Significado : (posición, velocidad, ángulo, vel. angular)")

# Aplicamos una acción arbitraria: empujar a la derecha
a = 1
obs_siguiente, recompensa, terminated, truncated, info = env.step(a)
print()
print(f"Acción aplicada: a = {a}")
print(f"Estado s_1     : {np.round(obs_siguiente, 4)}")
print(f"Recompensa r_1 : {recompensa}")
print(f"Terminado      : {terminated}")
print(f"Truncado       : {truncated}")

## 3. Primera política: el agente aleatorio

### 3.1. Definición formal

Una **política** $\pi$ es una distribución condicional sobre acciones dado el estado:
$$
\pi(a \mid s) \;=\; \mathbb{P}(A_t = a \mid S_t = s).
$$

La política aleatoria uniforme se define por
$$
\pi_{\text{rand}}(a \mid s) \;=\; \frac{1}{|\mathcal{A}|} \quad \forall\, s \in \mathcal{S},\; \forall\, a \in \mathcal{A}.
$$

Es la política más simple concebible: el agente ignora el estado y selecciona acciones equiprobablemente. Constituye una **línea de base** indispensable: cualquier algoritmo que diseñemos en clases sucesivas deberá superar consistentemente su rendimiento.

In [ ]:
def politica_aleatoria(estado, env: gym.Env, generador: np.random.Generator):
    '''
    Implementación de pi_rand: muestrea una acción uniformemente del espacio.
    El argumento `estado` se ignora deliberadamente (la política no depende de s).
    '''
    return int(generador.integers(low=0, high=env.action_space.n))


def rollout(env: gym.Env,
            politica: Callable,
            generador: np.random.Generator,
            max_pasos: int = 500) -> Dict:
    '''
    Ejecuta un episodio completo bajo la política dada.
    Devuelve un diccionario con la trayectoria y estadísticas básicas.
    '''
    estado, _ = env.reset(seed=int(generador.integers(0, 2**31 - 1)))
    trayectoria = {'estados': [estado], 'acciones': [], 'recompensas': []}

    for t in range(max_pasos):
        accion = politica(estado, env, generador)
        estado, r, terminated, truncated, _ = env.step(accion)

        trayectoria['acciones'].append(accion)
        trayectoria['recompensas'].append(r)
        trayectoria['estados'].append(estado)

        if terminated or truncated:
            break

    trayectoria['longitud'] = len(trayectoria['recompensas'])
    trayectoria['retorno_no_descontado'] = sum(trayectoria['recompensas'])
    return trayectoria


# --- Ejecución de un único episodio ---
gen = np.random.default_rng(SEMILLA)
traj = rollout(env, politica_aleatoria, gen)

print(f"Longitud del episodio   : {traj['longitud']} pasos")
print(f"Retorno (no descontado) : G_0 = {traj['retorno_no_descontado']}")
print()
print("Primeros 5 pasos de la trayectoria:")
for t in range(min(5, traj['longitud'])):
    s = np.round(traj['estados'][t], 3)
    a = traj['acciones'][t]
    r = traj['recompensas'][t]
    print(f"  t={t}: s={s}, a={a}, r={r}")

### 3.2. Cálculo del retorno descontado

El **retorno descontado** desde el paso $t$ se define
$$
G_t \;=\; \sum_{k=0}^{T-t-1} \gamma^k\, R_{t+k+1}.
$$

Implementemos su cálculo y comparemos para distintos valores de $\gamma$, ilustrando el papel del factor de descuento como ponderador temporal.

In [ ]:
def retorno_descontado(recompensas: List[float], gamma: float) -> float:
    '''Calcula G_0 = suma de gamma^k * R_{k+1} para k de 0 a T-1.'''
    return float(sum(gamma**k * r for k, r in enumerate(recompensas)))

# Comparemos distintos gamma sobre la misma trayectoria
print("Retorno descontado de la trayectoria anterior, para distintos gamma:")
print()
for g in [0.0, 0.5, 0.9, 0.99, 1.0]:
    G = retorno_descontado(traj['recompensas'], g)
    print(f"  gamma = {g:>4.2f}  ->  G_0 = {G:>8.4f}")

print()
print(f"Obsérvese que con gamma=1 recuperamos el retorno no descontado ({traj['retorno_no_descontado']:.0f}),")
print(f"mientras que con gamma=0 sólo cuenta la primera recompensa ({traj['recompensas'][0]}).")

## 4. Análisis estadístico de retornos bajo política aleatoria

Una sola trayectoria nos dice poco. La cantidad de interés teórica es el **valor de la política**:
$$
J(\pi) \;=\; \mathbb{E}_{\tau \sim \pi}[\, G_0 \,].
$$

No disponiendo de la dinámica $P$ en forma cerrada, **estimamos** $J(\pi)$ por el método de Monte Carlo: simulamos $N$ episodios independientes bajo la misma política y promediamos los retornos. Este es, en esencia, el primer algoritmo de RL que veremos —de manera más sistemática— en la Clase 3.

Por la **ley fuerte de los grandes números**, si los retornos $G_0^{(1)}, \ldots, G_0^{(N)}$ son i.i.d. con media finita,
$$
\hat{J}_N(\pi) \;=\; \frac{1}{N} \sum_{i=1}^{N} G_0^{(i)} \;\xrightarrow{c.s.}\; J(\pi).
$$

El **teorema central del límite** nos permite cuantificar la incertidumbre del estimador mediante un intervalo de confianza.

In [ ]:
# --- Ejecución masiva de episodios ---
N_EPISODIOS = 500
gen = np.random.default_rng(SEMILLA)

retornos = np.empty(N_EPISODIOS)
longitudes = np.empty(N_EPISODIOS, dtype=int)

for i in range(N_EPISODIOS):
    traj_i = rollout(env, politica_aleatoria, gen)
    retornos[i] = traj_i['retorno_no_descontado']
    longitudes[i] = traj_i['longitud']

# Estadísticas descriptivas
media = retornos.mean()
desv  = retornos.std(ddof=1)
ic95  = 1.96 * desv / np.sqrt(N_EPISODIOS)  # IC asintótico al 95%

print(f"=== Estimación de J(pi_rand) sobre {N_EPISODIOS} episodios ===")
print()
print(f"  J_N estimado    = {media:.3f}")
print(f"  IC 95%          = [{media - ic95:.3f}, {media + ic95:.3f}]")
print(f"  Desv. estándar  = {desv:.3f}")
print(f"  Mín / Máx        = {retornos.min():.0f} / {retornos.max():.0f}")
print(f"  Mediana          = {np.median(retornos):.1f}")

In [ ]:
# --- Visualización de la distribución empírica de retornos ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histograma
axes[0].hist(retornos, bins=30, color='#3D5A85', edgecolor='white', alpha=0.85)
axes[0].axvline(media, color='#D86A2C', lw=2, label=f'Media = {media:.1f}')
axes[0].axvline(np.median(retornos), color='#2E8A99', lw=2, linestyle='--',
                label=f'Mediana = {np.median(retornos):.1f}')
axes[0].set_xlabel(r'Retorno $G_0$')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución empírica de retornos\nbajo política aleatoria',
                  color='#1E3A5F')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Curva de convergencia del estimador
medias_acum = np.cumsum(retornos) / np.arange(1, N_EPISODIOS + 1)
axes[1].plot(medias_acum, color='#1E3A5F', lw=1.5)
axes[1].axhline(media, color='#D86A2C', lw=1, linestyle='--',
                label=f'Media final = {media:.2f}')
axes[1].set_xlabel(r'Número de episodios $N$')
axes[1].set_ylabel(r'$\hat{J}_N(\pi)$')
axes[1].set_title('Convergencia del estimador Monte Carlo',
                  color='#1E3A5F')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.1. Lectura estadística

Tres observaciones merecen comentario:

**1. Asimetría positiva.** El histograma exhibe sesgo a la derecha: la mayoría de los episodios fracasan rápidamente, pero ocasionalmente la dinámica casual del sistema y las acciones aleatorias coinciden en mantener el péndulo en pie por mayor tiempo. La distribución dista de ser gaussiana, hecho relevante al aplicar inferencia clásica.

**2. Variabilidad considerable.** La desviación estándar es del orden de la propia media. Esto evidencia que las políticas aleatorias son intrínsecamente ruidosas. Cualquier estimación con $N$ pequeño será imprecisa.

**3. Convergencia visible.** La curva acumulada de $\hat{J}_N$ se estabiliza con cientos de episodios. Para alcanzar precisión razonable necesitamos —en este problema concreto— al menos varios cientos de simulaciones, anticipo de un fenómeno general en RL: la **eficiencia de muestra** será una métrica central a lo largo del curso.

## 5. Anticipo: hacia el control óptimo

¿Cuán lejos está $\pi_{\text{rand}}$ del rendimiento óptimo? Con políticas razonables, CartPole-v1 admite retornos de hasta $500$ (el límite por truncamiento). Nuestra política aleatoria obtiene, típicamente, en torno a $20$–$25$ pasos. La brecha es enorme.

**Esto motiva las preguntas centrales de la Clase 2:**

1. ¿Cómo definimos formalmente la **función de valor** $V^\pi(s)$ asociada a una política?
2. ¿Cómo se vincula el valor con la **ecuación de Bellman** —y por qué dicha ecuación admite una solución única en condiciones razonables?
3. Cuando el espacio de estados-acciones es pequeño, ¿podemos enumerar y evaluar políticas de forma sistemática?
4. En problemas más simples (un solo estado, varias acciones), ¿cómo gestionamos el dilema **exploración–explotación**? Aquí emergerá el **bandido multibrazo**.

> **Tarea conceptual sugerida.** Antes del próximo encuentro, reflexione sobre lo siguiente: si dispusiera de la dinámica $P$ exacta del CartPole, ¿podría calcular $V^{\pi_{\text{rand}}}$ analíticamente? ¿Qué dificultad fundamental impide hacerlo en problemas reales?

## Ejercicios propuestos

> Estos ejercicios son optativos en la Clase 1, pero su resolución se valora en la participación. Plantee dudas en el foro de consultas.

### Ejercicio 1 — Estocasticidad en GridWorld

Modifique la clase `GridWorld` para introducir transiciones estocásticas: con probabilidad $p = 0.1$, la acción del agente "resbala" y se ejecuta una acción aleatoria de las cuatro posibles. Verifique que la probabilidad empírica $\hat{P}(s' \mid s, a)$, estimada sobre 10 000 transiciones desde un mismo $(s, a)$, converge a la probabilidad teórica.

### Ejercicio 2 — Retornos descontados y horizonte efectivo

Para CartPole-v1 bajo $\pi_{\text{rand}}$, estime $\mathbb{E}[G_0]$ con $N = 1000$ episodios para $\gamma \in \{0.5, 0.9, 0.95, 0.99, 1.0\}$. Reporte una tabla con media e intervalo de confianza al 95 %. Comente cómo varía la magnitud y la varianza del estimador con $\gamma$.

### Ejercicio 3 — Una política heurística simple

Diseñe una política heurística para CartPole basada únicamente en el ángulo del péndulo: si $\theta > 0$ empuje a la derecha; de lo contrario, a la izquierda. Estime $J(\pi_{\text{heuristica}})$ con $N = 500$ episodios. ¿Mejora respecto de la política aleatoria? Cuantifique la diferencia con un test estadístico apropiado (por ejemplo, t de Welch).

### Ejercicio 4 — Análisis de sesgo

¿Por qué fijamos `seed=SEMILLA` solamente en `reset()` y no en cada llamada a `step()`? ¿Qué sucedería si fijásemos la misma semilla en cada episodio? Justifique en términos de la independencia entre realizaciones requerida por el estimador Monte Carlo.

---

### Material complementario

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2.ª ed., 2018), Cap. 1 y 3.1–3.4.
- Gymnasium, [Documentación oficial](https://gymnasium.farama.org/).
- Repositorio del curso: notebooks Python y R, presentaciones, foro de consultas.

**Próximo encuentro:** martes 12 de mayo de 2026 — *Políticas, funciones de valor y el bandido multibrazo*.